# FRM Natation — visualisation

In [20]:
import json
from pathlib import Path
import pandas as pd
from setup_env import ensure_project_root

PROJECT_DIR = ensure_project_root()
HTML_RESULTS_DIR = (PROJECT_DIR / "data" / "processed" / "frmnatation" / "html_results").resolve()

## Chargement des données

In [21]:
def _normalize_swimmer(swimmer) -> dict:
    return swimmer if isinstance(swimmer, dict) else {}


def _competition_to_rows(comp: dict) -> list[dict]:
    """Une ligne par performance (épreuve × résultat)."""
    rows: list[dict] = []
    for epreuve in comp.get("epreuves") or []:
        if not isinstance(epreuve, dict):
            continue
        for perf in epreuve.get("performances") or []:
            if not isinstance(perf, dict):
                continue
            swimmer = _normalize_swimmer(perf.get("swimmer"))
            rows.append(
                {
                    "SwimDate": comp.get("SwimDate"),
                    "SwimYear": comp.get("SwimYear"),
                    "Meet": comp.get("Meet"),
                    "Location": comp.get("location"),
                    "Country": comp.get("Country"),
                    "Event": epreuve.get("Event"),
                    "Distance": epreuve.get("Distance"),
                    "Stroke": epreuve.get("Stroke"),
                    "Course": epreuve.get("Course"),
                    "PoolLength": epreuve.get("PoolLength"),
                    "Tour": epreuve.get("tour"),
                    "Rank": perf.get("Rank"),
                    "Club": perf.get("club"),
                    "SwimTime": perf.get("SwimTime"),
                    "SwimTimeSeconds": perf.get("SwimTimeSeconds"),
                    "Status": perf.get("Status"),
                    "Speed": perf.get("Speed"),
                    "Name": swimmer.get("Name"),
                    "Gender": swimmer.get("Gender"),
                    "Year_of_birth": swimmer.get("Year_of_birth"),
                    "Age": swimmer.get("Age"),
                    "AgeGroup": swimmer.get("AgeGroup"),
                    "Nationality": swimmer.get("Nationality"),
                }
            )
    return rows


def load_html_results(html_dir: Path = HTML_RESULTS_DIR) -> pd.DataFrame:
    """Charge tous les *.json sous html_results/ (aucun autre dossier)."""
    html_dir = html_dir.resolve()
    if html_dir != HTML_RESULTS_DIR:
        raise ValueError(
            f"Seul {HTML_RESULTS_DIR} est autorisé, reçu : {html_dir}"
        )
    if not html_dir.is_dir():
        raise FileNotFoundError(f"Dossier introuvable : {html_dir}")

    all_rows: list[dict] = []
    for path in sorted(html_dir.glob("*.json")):
        payload = json.loads(path.read_text(encoding="utf-8"))
        if isinstance(payload, dict):
            all_rows.extend(_competition_to_rows(payload))
    return pd.DataFrame(all_rows)

In [22]:
json_files = sorted(HTML_RESULTS_DIR.glob("*.json"))
df = load_html_results(HTML_RESULTS_DIR)

print("\nNombre de lignes :", len(df))
print("Nombre de compétitions (fichiers) :", len(json_files))


Nombre de lignes : 20283
Nombre de compétitions (fichiers) : 43


In [23]:
def empty_mask(series: pd.Series) -> pd.Series:
    """Masque True pour les valeurs vides (NaN, None ou chaîne vide)."""
    return series.isna() | (series.astype(str).str.strip() == "")


def count_empty_rows(df: pd.DataFrame, column: str) -> int:
    """Nombre de lignes vides pour une colonne."""
    return int(empty_mask(df[column]).sum())

## Colonnes du DataFrame

In [24]:
print(f"Nombre de colonnes : {len(df.columns)}\n")
print("Colonnes :")
for i, col in enumerate(df.columns, start=1):
    print(f"  {i:2d}. {col}")

Nombre de colonnes : 23

Colonnes :
   1. SwimDate
   2. SwimYear
   3. Meet
   4. Location
   5. Country
   6. Event
   7. Distance
   8. Stroke
   9. Course
  10. PoolLength
  11. Tour
  12. Rank
  13. Club
  14. SwimTime
  15. SwimTimeSeconds
  16. Status
  17. Speed
  18. Name
  19. Gender
  20. Year_of_birth
  21. Age
  22. AgeGroup
  23. Nationality


## Première ligne

In [25]:
df.head(1)

,SwimDate,SwimYear,Meet,Location,Country,Event,Distance,Stroke,Course,PoolLength,...,SwimTime,SwimTimeSeconds,Status,Speed,Name,Gender,Year_of_birth,Age,AgeGroup,Nationality
0,2016-07-24,2016,CHAMPIONNATS DU MAROC M C J S ET OPEN - CASABL...,,MAR,50 FR SCM,50,FR,SCM,25,...,28.14,28.14,OK,1.7768,MANA Noura,F,1997,19,19 & Over,MAR


## Vérification de `SwimTimeSeconds` 

In [26]:
col = "SwimTimeSeconds"
series = df[col]
is_missing = empty_mask(series)
n_empty = count_empty_rows(df, col)

numeric = pd.to_numeric(series, errors="coerce")
is_non_numeric = numeric.isna() & ~is_missing

invalid = is_missing | is_non_numeric
n_total = len(df)
n_invalid = int(invalid.sum())
n_valid = n_total - n_invalid

print(f"SwimTimeSeconds valides (non vide + numérique) : {n_valid}")
print(f"  — manquantes/vides  : {n_empty}")
print(f"  — non numériques    : {int(is_non_numeric.sum())}")

SwimTimeSeconds valides (non vide + numérique) : 20283
  — manquantes/vides  : 0
  — non numériques    : 0


## Vérification de `Name` 

In [27]:
col = "Name"
series = df[col]

n_empty = count_empty_rows(df, col)
is_missing = empty_mask(series)

stripped = series.astype(str).str.strip()
numeric = pd.to_numeric(stripped, errors="coerce")
is_numeric = (~is_missing) & numeric.notna()

print(f"Name vides              : {n_empty}")
print(f"Name à valeur numérique : {int(is_numeric.sum())}")

Name vides              : 0
Name à valeur numérique : 0


## Vérification de `Event` 

In [75]:
col = "Event"
n_empty = count_empty_rows(df, col)
n_total = len(df)
n_valid = n_total - n_empty

distinct_events = (
    df[col]
    .dropna()
    .astype(str)
    .str.strip()
)
distinct_events = sorted(distinct_events[distinct_events != ""].unique())

print(f"Event valides (non vide) : {n_valid}")
print(f"  — manquantes/vides  : {n_empty}")
print(f"\nEvent distincts ({len(distinct_events)}) :")
for event in distinct_events:
    print(f"  - {event}")

Event valides (non vide) : 20283
  — manquantes/vides  : 0

Event distincts (36) :
  - 100 BK LCM
  - 100 BK SCM
  - 100 BR LCM
  - 100 BR SCM
  - 100 FL LCM
  - 100 FL SCM
  - 100 FR LCM
  - 100 FR SCM
  - 1500 FR SCM
  - 200 BK LCM
  - 200 BK SCM
  - 200 BR LCM
  - 200 BR SCM
  - 200 FL LCM
  - 200 FL SCM
  - 200 FR SCM
  - 200 IM LCM
  - 200 IM SCM
  - 25 BK SCM
  - 25 BR SCM
  - 25 FL SCM
  - 25 FR SCM
  - 400 FR SCM
  - 400 IM SCM
  - 400 REL LCM
  - 400 REL SCM
  - 50 BK LCM
  - 50 BK SCM
  - 50 BR LCM
  - 50 BR SCM
  - 50 FL LCM
  - 50 FL SCM
  - 50 FR LCM
  - 50 FR SCM
  - 800 FR SCM
  - 800 REL SCM


## Vérification de `Age` 

In [76]:
col = "Age"
series = df[col]

is_missing = empty_mask(series)
n_empty = count_empty_rows(df, col)

numeric = pd.to_numeric(series, errors="coerce")
is_non_numeric = numeric.isna() & ~is_missing
is_non_positive = (~is_missing) & (~is_non_numeric) & (numeric <= 0)

invalid = is_missing | is_non_numeric | is_non_positive
n_total = len(df)
n_invalid = int(invalid.sum())
n_valid = n_total - n_invalid

print(f"Age valides (non vide + numérique + > 0) : {n_valid}")
print(f"  — manquantes/vides  : {n_empty}")
print(f"  — non numériques    : {int(is_non_numeric.sum())}")
print(f"  — non positifs (≤ 0): {int(is_non_positive.sum())}")

Age valides (non vide + numérique + > 0) : 20283
  — manquantes/vides  : 0
  — non numériques    : 0
  — non positifs (≤ 0): 0


## Recherche par `Name` et `Event`

Recherche dans toutes les performances chargées depuis `data/processed/frmnatation/html_results` (tous les fichiers `*.json`), en filtrant sur le nageur (`Name`) et l'épreuve (`Event`).

In [40]:
def _match_column(
    series: pd.Series,
    query: str,
    *,
    exact: bool,
    case_insensitive: bool,
) -> pd.Series:
    values = series.astype(str).str.strip()
    q = str(query).strip()
    if case_insensitive:
        values_cmp = values.str.casefold()
        q_cmp = q.casefold()
    else:
        values_cmp = values
        q_cmp = q
    if exact:
        return values_cmp == q_cmp
    return values_cmp.str.contains(q_cmp, regex=False, na=False)


def search_by_name_and_event(
    name_query: str,
    event_query: str,
    *,
    data: pd.DataFrame | None = None,
    exact: bool = False,
    case_insensitive: bool = True,
) -> pd.DataFrame:
    """Filtre les lignes selon `Name` et `Event`."""
    if not str(name_query).strip():
        raise ValueError("Indiquez un nom (name_query non vide).")
    if not str(event_query).strip():
        raise ValueError("Indiquez une épreuve (event_query non vide).")

    scoped = data if data is not None else df
    if scoped.empty:
        return scoped.iloc[0:0].copy()
    for col in ("Name", "Event"):
        if col not in scoped.columns:
            return scoped.iloc[0:0].copy()

    mask = _match_column(
        scoped["Name"],
        name_query,
        exact=exact,
        case_insensitive=case_insensitive,
    ) & _match_column(
        scoped["Event"],
        event_query,
        exact=exact,
        case_insensitive=case_insensitive,
    )
    return scoped.loc[mask].copy()


SEARCH_NAME = "azize othmane"
SEARCH_EVENT = "100 BK SCM"  
SEARCH_EXACT = False  

results = search_by_name_and_event(
    SEARCH_NAME,
    SEARCH_EVENT,
    exact=SEARCH_EXACT,
)

mode = "exact" if SEARCH_EXACT else "contient"
print(f"Recherche Name  : « {SEARCH_NAME.strip()} » ({mode})")
print(f"Recherche Event : « {SEARCH_EVENT.strip()} » ({mode})")
print(f"Lignes trouvées : {len(results)}")
if not results.empty:
    distinct_names = sorted(
        results["Name"].astype(str).str.strip().unique(), key=str.casefold
    )
    distinct_events = sorted(
        results["Event"].astype(str).str.strip().unique(), key=str.casefold
    )
    print(f"Noms distincts ({len(distinct_names)}) :")
    for name in distinct_names[:10]:
        print(f"  - {name}")
    if len(distinct_names) > 10:
        print(f"  … et {len(distinct_names) - 10} autre(s)")
    print(f"Events distincts ({len(distinct_events)}) :")
    for event in distinct_events[:10]:
        print(f"  - {event}")
    if len(distinct_events) > 10:
        print(f"  … et {len(distinct_events) - 10} autre(s)")

display_cols = [
    "SwimDate",
    "Meet",
    "Event",
    "Tour",
    "Rank",
    "Name",
    "Age",
    "Year_of_birth",
    "Club",
    "SwimTime",
    "SwimTimeSeconds",
]
results[display_cols].sort_values(
    ["Name", "Event", "SwimDate", "SwimTimeSeconds"],
    na_position="last",
)

Recherche Name  : « azize othmane » (contient)
Recherche Event : « 100 BK SCM » (contient)
Lignes trouvées : 2
Noms distincts (1) :
  - AZIZE Othmane
Events distincts (1) :
  - 100 BK SCM


,SwimDate,Meet,Event,Tour,Rank,Name,Age,Year_of_birth,Club,SwimTime,SwimTimeSeconds
11305,2016-05-13,INTERCLUBS REGIONAUX 3 - CASABLANCA - CASABLAN...,100 BK SCM,CADETS Séries,8.0,AZIZE Othmane,16,2000,WAC,1:18.25,78.25
16366,2016-05-15,INTERCLUBS REGIONAUX 4 - CASABLANCA - CASABLAN...,100 BK SCM,CADETS Séries,7.0,AZIZE Othmane,16,2000,WAC,1:17.75,77.75


## Vérification de `Gender`

In [77]:
col = "Gender"
series = df[col]

n_empty = count_empty_rows(df, col)
is_missing = empty_mask(series)

gender = series.astype(str).str.strip().str.upper()
is_valid = (~is_missing) & gender.isin(["F", "M"])

n_valid = int(is_valid.sum())

print(f"Gender valides (F ou M) : {n_valid}")
print(f"  — manquantes/vides  : {n_empty}")
print(f"  — autres valeurs    : {int((~is_missing & ~is_valid).sum())}")

Gender valides (F ou M) : 20283
  — manquantes/vides  : 0
  — autres valeurs    : 0
